#### ============================================================
#### Fake News Detection - Final Project Code V4
#### Author: Xiaoxi Gao & Zhenzhe Luo
####
#### Goal:
#### Build a reproducible and interpretable traditional machine
#### learning system to classify fake vs true news articles.
####
#### Dataset:
#### - Original Fake.csv and True.csv dataset
#### - Label definition: Fake = 0, True = 1
#### - Main model input: article text only
#### - The subject and date columns are kept only for bias/data
####   analysis and are not used as model features.
####
#### Standard pipeline:
#### 1. Load Fake.csv and True.csv
#### 2. Build text-only input feature
#### 3. Clean text
#### 4. Stratified train/test split
#### 5. TF-IDF vectorization inside each pipeline
#### 6. Cross-validation training using 5-fold StratifiedKFold
#### 7. Hyperparameter tuning using GridSearchCV
#### 8. Model comparison using Macro F1, Fake F1, and True F1
#### 9. Select the best model based on CV Macro F1
#### 10. Retrain/refit the selected best model on the full training data
#### 11. Final evaluation on the held-out test set using Accuracy,
####     Fake F1, True F1, Macro F1, Weighted F1, AUC, ROC curve,
####     and confusion matrix
####
#### Model candidates:
#### - Baseline model: DummyClassifier
#### - Traditional text classifiers: MultinomialNB,
####   Logistic Regression, and LinearSVC
#### - Tree-based ensemble model: RandomForest
#### - Ensemble learning extension: StackingClassifier
####
#### Instructor-required components:
#### - Baseline solution
#### - Cross validation and hyperparameter tuning
#### - Statistical component: bootstrapping confidence intervals
#### - Interpretable results: TF-IDF feature inspection and model
####   coefficient analysis
#### - Ensemble learning: RandomForest and StackingClassifier
#### - Bias/generalization analysis: subject bias check and
####   discussion of same-dataset performance
####
#### Important note:
#### The best model is selected using cross-validation performance
#### on the training set, not by test-set performance. The held-out
#### test set is used only once for final evaluation.
#### ============================================================

In [1]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

#### ============================================================
#### 0. Configuration
#### ============================================================

In [2]:
FAKE_PATH = "Fake.csv"
TRUE_PATH = "True.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
BOOTSTRAP_ROUNDS = 300

OUTPUT_DIR = "final_results_fake_news_v4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

#### ============================================================
####  Utility functions
####  ============================================================

In [3]:
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def save_text_report(text, filename):
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        f.write(text)


def clean_text(text):
    """
    Clean English news text.
    1. Remove URLs
    2. Remove HTML tags
    3. Lowercase
    4. Remove non-letter characters
    5. Remove extra spaces
    """
    text = str(text)
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


    
def build_text_column(df, mode="text_only"):
    """
    Build text features for fake news classification.

    Available modes:
    - text_only: use only article text
    - title_text: use title + text
    - subject_title_text: use subject + title + text

    Recommended main experiment:
    - text_only or title_text

    Not recommended for main experiment:
    - subject_title_text, because subject may introduce dataset-specific shortcuts.
    """
    title_col = None
    text_col = None
    subject_col = None

    for c in df.columns:
        lc = c.lower()
        if lc == "title":
            title_col = c
        elif lc == "text":
            text_col = c
        elif lc == "subject":
            subject_col = c

    parts = []

    if mode == "text_only":
        if text_col is not None:
            parts.append(df[text_col].fillna("").astype(str))

    elif mode == "title_text":
        if title_col is not None:
            parts.append(df[title_col].fillna("").astype(str))
        if text_col is not None:
            parts.append(df[text_col].fillna("").astype(str))

    elif mode == "subject_title_text":
        if subject_col is not None:
            parts.append(df[subject_col].fillna("").astype(str))
        if title_col is not None:
            parts.append(df[title_col].fillna("").astype(str))
        if text_col is not None:
            parts.append(df[text_col].fillna("").astype(str))

    else:
        raise ValueError("mode must be 'text_only', 'title_text', or 'subject_title_text'.")

    if len(parts) == 0:
        raise ValueError("No usable text columns found for the selected mode.")

    combined = parts[0]
    for p in parts[1:]:
        combined = combined + " " + p

    return combined


def bootstrap_metric_ci(y_true, y_pred, metric_func, n_rounds=300, random_state=42):
    """
    Bootstrap 95% confidence interval.
    This is the statistical component required by the instructor.
    """
    rng = np.random.default_rng(random_state)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)

    scores = []
    for _ in range(n_rounds):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        scores.append(metric_func(yt, yp))

    mean_score = np.mean(scores)
    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)

    return mean_score, lower, upper


def get_model_scores(model, X):
    """
    Return continuous scores for AUC/ROC.
    Prefer decision_function, otherwise predict_proba.
    """
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    elif hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    else:
        return None


#### ============================================================
#### STEP 1 - Load data
#### ============================================================

In [4]:
# print_section("STEP 1 - Load Data")

# MERGED_PATH = "merged_welfake_fake_true_final.csv"

# df = pd.read_csv(
#     MERGED_PATH,
#     sep="\t",
#     dtype=str,
#     encoding="utf-8",
#     on_bad_lines="skip"
# )

# df = df[["title", "text", "label"]].copy()
# df = df.dropna(subset=["text"]).copy()
# df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# df["label"] = df["label"].astype(int)

# FEATURE_MODE = "text_only"

# df["combined_text"] = build_text_column(df, mode=FEATURE_MODE)

# df = df[["combined_text", "label", "title", "text"]].copy()

# print("Feature mode:", FEATURE_MODE)
# print("Model input column: combined_text")
# print("Merged dataset shape:", df.shape)

# print("\nLabel distribution:")
# print(df["label"].value_counts())

# print("\nLabel ratio:")
# print(df["label"].value_counts(normalize=True).round(4))

# print("\nExample combined_text:")
# print(df["combined_text"].iloc[0][:500])

In [5]:
print_section("STEP 1 - Load Data")

fake_df = pd.read_csv(FAKE_PATH)
true_df = pd.read_csv(TRUE_PATH)

# Label definition: Fake = 0, True = 1
fake_df["label"] = 0
true_df["label"] = 1

# Build combined text
# FEATURE_MODE options:
# - text_only: use only text
# - title_text: use title + text
# - subject_title_text: use subject + title + text
FEATURE_MODE = "text_only"

fake_df["combined_text"] = build_text_column(fake_df, mode=FEATURE_MODE)
true_df["combined_text"] = build_text_column(true_df, mode=FEATURE_MODE)

# Merge Fake and True datasets
df = pd.concat([fake_df, true_df], ignore_index=True)

# Keep subject and date only for analysis, not for model input
keep_cols = ["combined_text", "label"]

if "subject" in df.columns:
    keep_cols.append("subject")

if "date" in df.columns:
    keep_cols.append("date")

df = df[keep_cols].copy()

print("Feature mode:", FEATURE_MODE)
print("Model input column: combined_text")
print("Original dataset shape:", df.shape)

print("\nOriginal label distribution:")
print(df["label"].value_counts())

print("\nOriginal label ratio:")
print(df["label"].value_counts(normalize=True).round(4))

print("\nExample combined_text:")
print(df["combined_text"].iloc[0][:500])


STEP 1 - Load Data
Feature mode: text_only
Model input column: combined_text
Original dataset shape: (44898, 4)

Original label distribution:
label
0    23481
1    21417
Name: count, dtype: int64

Original label ratio:
label
0    0.523
1    0.477
Name: proportion, dtype: float64

Example combined_text:
Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a gr


#### ============================================================
#### STEP 2 - Clean text
#### ============================================================

In [6]:
print_section("STEP 2 - Clean Text")

df["combined_text"] = df["combined_text"].astype(str).apply(clean_text)

# Remove empty texts
df = df[df["combined_text"].str.len() > 0]

# Remove duplicates
df = df.drop_duplicates(subset=["combined_text"]).reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)
print("Label distribution after cleaning:")
print(df["label"].value_counts())


STEP 2 - Clean Text
Dataset shape after cleaning: (38313, 4)
Label distribution after cleaning:
label
1    20921
0    17392
Name: count, dtype: int64


#### ============================================================
#### STEP 3 - Train/Test split
#### ============================================================

In [7]:
print_section("STEP 3 - Train/Test Split")

X = df["combined_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


STEP 3 - Train/Test Split
Training samples: 30650
Testing samples : 7663


#### ============================================================
#### STEP 4 - TF-IDF Vectorization
#### TF-IDF stays inside the pipeline to avoid data leakage.
#### ============================================================

In [8]:
print_section("STEP 4 - TF-IDF Vectorization")

print("TF-IDF is included inside each pipeline to avoid data leakage.")
#max_features = 5000, 10000, 20000, 30000
tfidf = TfidfVectorizer(stop_words="english",
    lowercase=True,
    strip_accents="unicode",
    max_df=0.85,
    min_df=3,
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True
)


STEP 4 - TF-IDF Vectorization
TF-IDF is included inside each pipeline to avoid data leakage.


In [9]:
print_section("STEP 4.1 - Tokenization and TF-IDF Table Demo")

# This cell is only for demonstration.
# The real model training still uses TF-IDF inside the pipeline to avoid data leakage.

# =========================
# Part 1: Tokenization table
# =========================

demo_n = 5   # show first 5 samples only

analyzer = tfidf.build_analyzer()

token_demo_df = pd.DataFrame({
    "cleaned_text": X_train.iloc[:demo_n].values
})

token_demo_df["tokens_or_ngrams"] = token_demo_df["cleaned_text"].apply(
    lambda x: analyzer(x)[:40]
)

token_demo_df["number_of_tokens_or_ngrams"] = token_demo_df["cleaned_text"].apply(
    lambda x: len(analyzer(x))
)

# Shorten text for display
token_demo_df["cleaned_text_preview"] = token_demo_df["cleaned_text"].apply(
    lambda x: x[:300] + "..." if len(x) > 300 else x
)

token_demo_df = token_demo_df[
    ["cleaned_text_preview", "tokens_or_ngrams", "number_of_tokens_or_ngrams"]
]

print("Tokenization Demo Table:")
display(token_demo_df)


STEP 4.1 - Tokenization and TF-IDF Table Demo
Tokenization Demo Table:


,cleaned_text_preview,tokens_or_ngrams,number_of_tokens_or_ngrams
0,bucharest reuters romania s lower house of par...,"[bucharest, reuters, romania, lower, house, pa...",383
1,washington reuters the united states wants pak...,"[washington, reuters, united, states, wants, p...",425
2,the hosts of fox friends were hoping a minneso...,"[hosts, fox, friends, hoping, minnesota, polic...",593
3,if you didn t know friday was holocaust rememb...,"[didn, know, friday, holocaust, remembrance, d...",883
4,as we all live in the nightmarish hellscape th...,"[live, nightmarish, hellscape, donald, trump, ...",267


In [10]:
# =========================
# Part 2: TF-IDF matrix table
# =========================

# Fit TF-IDF on training data only for demonstration.
# This matrix is NOT used for final model training.
X_train_tfidf_demo = tfidf.fit_transform(X_train)

feature_names = tfidf.get_feature_names_out()

print("TF-IDF matrix shape:", X_train_tfidf_demo.shape)
print("Rows = training samples; Columns = TF-IDF features.")

# Display only a small matrix to avoid memory/display issues
show_rows = 10
show_cols = 30

tfidf_demo_table = pd.DataFrame(
    X_train_tfidf_demo[:show_rows, :show_cols].toarray(),
    columns=feature_names[:show_cols]
)

print(f"TF-IDF Demo Matrix: first {show_rows} samples x first {show_cols} features")
display(tfidf_demo_table)

TF-IDF matrix shape: (30650, 10000)
Rows = training samples; Columns = TF-IDF features.
TF-IDF Demo Matrix: first 10 samples x first 30 features


,aaron,abadi,abandon,abandoned,abandoning,abbas,abbott,abc,abc news,abc week,...,abortion,abortions,abroad,abruptly,absence,absent,absolute,absolutely,absurd,abu
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.04912,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# =========================
# Part 3: Non-zero TF-IDF features for one sample
# =========================

sample_index = 0
sample_vector = X_train_tfidf_demo[sample_index]

nonzero_indices = sample_vector.nonzero()[1]
nonzero_values = sample_vector.data

sample_tfidf_df = pd.DataFrame({
    "feature": feature_names[nonzero_indices],
    "tfidf_value": nonzero_values
}).sort_values(by="tfidf_value", ascending=False)

print("Top 20 non-zero TF-IDF features for sample 0:")
display(sample_tfidf_df.head(20))

# Save outputs
token_demo_df.to_csv(
    os.path.join(OUTPUT_DIR, "tokenization_demo_table.csv"),
    index=False
)

tfidf_demo_table.to_csv(
    os.path.join(OUTPUT_DIR, "tfidf_demo_matrix_table.csv"),
    index=False
)

sample_tfidf_df.head(50).to_csv(
    os.path.join(OUTPUT_DIR, "sample0_top_tfidf_features.csv"),
    index=False
)

# Clean up demo variable to avoid confusion
del X_train_tfidf_demo

Top 20 non-zero TF-IDF features for sample 0:


,feature,tfidf_value
137,social democrats,0.185005
20,bills,0.177909
1,romania,0.175362
133,street protests,0.168168
5,approved,0.167154
69,prosecutors,0.151004
129,lower house,0.149027
131,european commission,0.146276
23,triggered,0.135588
10,european,0.131437


#### ============================================================
#### STEP 5 - Cross validation training
#### STEP 6 - Hyperparameter tuning (GridSearchCV)
#### ============================================================

In [12]:
print_section("STEP 5 & STEP 6 - Cross Validation + Hyperparameter Tuning")

import warnings
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

# Suppress non-critical warnings to make notebook output cleaner
warnings.filterwarnings("ignore")

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# Important label meaning:
# 0 = Fake News
# 1 = True News
#
# sklearn's default binary F1 uses positive label = 1.
# That means default F1 mainly evaluates the True class.
# To avoid confusion, we explicitly report:
# - Fake F1
# - True F1
# - Macro F1
# - Weighted F1
# ------------------------------------------------------------

def evaluate_model(y_true, y_pred):
    """
    Compute evaluation metrics for fake/true news classification.

    Label meaning:
    0 = Fake
    1 = True
    """
    return {
        "Test_Accuracy": accuracy_score(y_true, y_pred),

        # Fake class metrics, label 0
        "Test_Precision_Fake": precision_score(
            y_true, y_pred, pos_label=0, zero_division=0
        ),
        "Test_Recall_Fake": recall_score(
            y_true, y_pred, pos_label=0, zero_division=0
        ),
        "Test_F1_Fake": f1_score(
            y_true, y_pred, pos_label=0, zero_division=0
        ),

        # True class metrics, label 1
        "Test_Precision_True": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "Test_Recall_True": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "Test_F1_True": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),

        # Overall metrics
        "Test_F1_Macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Test_F1_Weighted": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
    }


# ------------------------------------------------------------
# Model candidates
# Baseline_Dummy: majority-class baseline
# MultinomialNB: classical text classification baseline
# LogisticRegression: interpretable linear model
# LinearSVC: strong sparse text classifier
# RandomForest: nonlinear ensemble baseline
# Stacking: ensemble model combining multiple classifiers
# ------------------------------------------------------------

model_configs = {
    "Baseline_Dummy": {
        "model": DummyClassifier(strategy="most_frequent"),
        "param_grid": None
    },

    "MultinomialNB": {
        "model": MultinomialNB(),
        "param_grid": {
            "clf__alpha": [0.1, 0.5, 1.0]
        }
    },

    "LogisticRegression": {
        "model": LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__C": [0.1, 1.0, 10.0]
        }
    },

    "LinearSVC": {
        "model": LinearSVC(
            random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__C": [0.1, 1.0, 10.0]
        }
    },

    "RandomForest": {
        "model": RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "param_grid": {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [None, 20]
        }
    },

    "Stacking": {
        "model": StackingClassifier(
            estimators=[
                (
                    "lr",
                    LogisticRegression(
                        C=1.0,
                        max_iter=3000,
                        random_state=RANDOM_STATE
                    )
                ),
                (
                    "nb",
                    MultinomialNB(alpha=0.5)
                ),
                (
                    "svm",
                    LinearSVC(
                        C=1.0,
                        random_state=RANDOM_STATE
                    )
                ),
                (
                    "rf",
                    RandomForestClassifier(
                        n_estimators=100,
                        max_depth=None,
                        random_state=RANDOM_STATE,
                        n_jobs=-1
                    )
                )
            ],
            final_estimator=LogisticRegression(
                max_iter=3000,
                random_state=RANDOM_STATE
            ),
            cv=5,
            n_jobs=-1
        ),
        "param_grid": {
            "clf__final_estimator__C": [0.1, 1.0, 10.0]
        }
    }
}

results = []
best_estimators = {}

# Use macro F1 for model selection.
# Macro F1 treats Fake and True classes equally.
SCORING = "f1_macro"

for model_name, config in model_configs.items():
    print(f"\n>>> Running model: {model_name}")

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("clf", config["model"])
    ])

    # --------------------------------------------------------
    # Baseline model: no hyperparameter tuning
    # --------------------------------------------------------
    if model_name == "Baseline_Dummy":

        # Real cross-validation score for DummyClassifier
        cv_scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            scoring=SCORING,
            cv=cv,
            n_jobs=-1
        )

        pipeline.fit(X_train, y_train)
        y_pred_test = pipeline.predict(X_test)

        metric_dict = evaluate_model(y_test, y_pred_test)

        row = {
            "Model": model_name,
            "Best_Params": "N/A",
            "CV_F1_Macro_Mean": cv_scores.mean(),
            "CV_F1_Macro_Std": cv_scores.std()
        }

        row.update(metric_dict)

        results.append(row)
        best_estimators[model_name] = pipeline

        print("Best Params:", row["Best_Params"])
        print("CV Macro F1 Mean :", round(row["CV_F1_Macro_Mean"], 4))
        print("CV Macro F1 Std  :", round(row["CV_F1_Macro_Std"], 4))
        print("Test Fake F1     :", round(row["Test_F1_Fake"], 4))
        print("Test True F1     :", round(row["Test_F1_True"], 4))
        print("Test Macro F1    :", round(row["Test_F1_Macro"], 4))
        print("Test Weighted F1 :", round(row["Test_F1_Weighted"], 4))

    # --------------------------------------------------------
    # Other models: GridSearchCV for hyperparameter tuning
    # --------------------------------------------------------
    else:
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=config["param_grid"],
            scoring=SCORING,
            cv=cv,
            n_jobs=-1,
            refit=True
        )

        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        y_pred_test = best_model.predict(X_test)

        metric_dict = evaluate_model(y_test, y_pred_test)

        row = {
            "Model": model_name,
            "Best_Params": grid.best_params_,
            "CV_F1_Macro_Mean": grid.best_score_,
            "CV_F1_Macro_Std": grid.cv_results_["std_test_score"][grid.best_index_]
        }

        row.update(metric_dict)

        results.append(row)
        best_estimators[model_name] = best_model

        print("Best Params:", grid.best_params_)
        print("CV Macro F1 Mean :", round(row["CV_F1_Macro_Mean"], 4))
        print("CV Macro F1 Std  :", round(row["CV_F1_Macro_Std"], 4))
        print("Test Fake F1     :", round(row["Test_F1_Fake"], 4))
        print("Test True F1     :", round(row["Test_F1_True"], 4))
        print("Test Macro F1    :", round(row["Test_F1_Macro"], 4))
        print("Test Weighted F1 :", round(row["Test_F1_Weighted"], 4))


# Convert results to DataFrame
results_df = pd.DataFrame(results)

print_section("MODEL RESULTS SUMMARY")

display_cols = [
    "Model",
    "Best_Params",
    "CV_F1_Macro_Mean",
    "CV_F1_Macro_Std",
    "Test_Accuracy",
    "Test_Precision_Fake",
    "Test_Recall_Fake",
    "Test_F1_Fake",
    "Test_Precision_True",
    "Test_Recall_True",
    "Test_F1_True",
    "Test_F1_Macro",
    "Test_F1_Weighted"
]

results_df = results_df[display_cols].sort_values(
    by="Test_F1_Macro",
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

# Save model comparison table
results_df.to_csv(
    os.path.join(OUTPUT_DIR, "model_results_macro_f1_summary.csv"),
    index=False
)

print("\nSaved model results to:")
print(os.path.join(OUTPUT_DIR, "model_results_macro_f1_summary.csv"))


STEP 5 & STEP 6 - Cross Validation + Hyperparameter Tuning

>>> Running model: Baseline_Dummy


/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version 

Best Params: N/A
CV Macro F1 Mean : 0.3532
CV Macro F1 Std  : 0.0
Test Fake F1     : 0.0
Test True F1     : 0.7063
Test Macro F1    : 0.3532
Test Weighted F1 : 0.3857

>>> Running model: MultinomialNB


/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version 

Best Params: {'clf__alpha': 0.1}
CV Macro F1 Mean : 0.9572
CV Macro F1 Std  : 0.0034
Test Fake F1     : 0.9481
Test True F1     : 0.9573
Test Macro F1    : 0.9527
Test Weighted F1 : 0.9531

>>> Running model: LogisticRegression
Best Params: {'clf__C': 10.0}
CV Macro F1 Mean : 0.9933
CV Macro F1 Std  : 0.0005
Test Fake F1     : 0.9924
Test True F1     : 0.9937
Test Macro F1    : 0.993
Test Weighted F1 : 0.9931

>>> Running model: LinearSVC
Best Params: {'clf__C': 1.0}
CV Macro F1 Mean : 0.9949
CV Macro F1 Std  : 0.0005
Test Fake F1     : 0.9944
Test True F1     : 0.9954
Test Macro F1    : 0.9949
Test Weighted F1 : 0.9949

>>> Running model: RandomForest
Best Params: {'clf__max_depth': None, 'clf__n_estimators': 200}
CV Macro F1 Mean : 0.9959
CV Macro F1 Std  : 0.0007
Test Fake F1     : 0.9957
Test True F1     : 0.9964
Test Macro F1    : 0.996
Test Weighted F1 : 0.9961

>>> Running model: Stacking
Best Params: {'clf__final_estimator__C': 10.0}
CV Macro F1 Mean : 0.9974
CV Macro F1 Std  :

,Model,Best_Params,CV_F1_Macro_Mean,CV_F1_Macro_Std,Test_Accuracy,Test_Precision_Fake,Test_Recall_Fake,Test_F1_Fake,Test_Precision_True,Test_Recall_True,Test_F1_True,Test_F1_Macro,Test_F1_Weighted
0,Stacking,{'clf__final_estimator__C': 10.0},0.9974,0.0002,0.9979,0.9991,0.9963,0.9977,0.9969,0.9993,0.9981,0.9979,0.9979
1,RandomForest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.9959,0.0007,0.9961,0.9983,0.9931,0.9957,0.9943,0.9986,0.9964,0.9960,0.9961
2,LinearSVC,{'clf__C': 1.0},0.9949,0.0005,0.9949,0.9977,0.9911,0.9944,0.9926,0.9981,0.9954,0.9949,0.9949
3,LogisticRegression,{'clf__C': 10.0},0.9933,0.0005,0.9931,0.9965,0.9882,0.9924,0.9903,0.9971,0.9937,0.9930,0.9931
4,MultinomialNB,{'clf__alpha': 0.1},0.9572,0.0034,0.9532,0.9545,0.9416,0.9481,0.9520,0.9627,0.9573,0.9527,0.9531
5,Baseline_Dummy,N/A,0.3532,0.0000,0.5460,0.0000,0.0000,0.0000,0.5460,1.0000,0.7063,0.3532,0.3857



Saved model results to:
final_results_fake_news_v4/model_results_macro_f1_summary.csv


#### ============================================================
#### STEP 7 - Model comparison (F1)
#### IMPORTANT: choose model by CV_F1_Mean, not Test_F1.
#### ============================================================

In [16]:
print_section("STEP 7 - Model Comparison and Best Model Selection")

# results_df has already been created in STEP 5 & 6.
# Here we select the best model using CV Macro F1, not Test F1.
# This avoids using the test set for model selection.

results_df_nonbaseline = results_df[
    results_df["Model"] != "Baseline_Dummy"
].copy()

results_df_nonbaseline = results_df_nonbaseline.sort_values(
    by="CV_F1_Macro_Mean",
    ascending=False
).reset_index(drop=True)

print("All model results:")
display(results_df.round(4))

print("\nRanked models by CV Macro F1:")
display(results_df_nonbaseline.round(4))

# Select best model based on training-set cross-validation performance
best_model_name = results_df_nonbaseline.iloc[0]["Model"]
best_model = best_estimators[best_model_name]

print("\nBest model selected by CV Macro F1:", best_model_name)

display(
    results_df_nonbaseline[
        results_df_nonbaseline["Model"] == best_model_name
    ][
        [
            "Model",
            "CV_F1_Macro_Mean",
            "CV_F1_Macro_Std",
            "Test_Accuracy",
            "Test_F1_Fake",
            "Test_F1_True",
            "Test_F1_Macro",
            "Test_F1_Weighted"
        ]
    ].round(4)
)

results_df.to_csv(
    os.path.join(OUTPUT_DIR, "all_model_results.csv"),
    index=False
)

results_df_nonbaseline.to_csv(
    os.path.join(OUTPUT_DIR, "ranked_model_results_by_cv_macro_f1.csv"),
    index=False
)


STEP 7 - Model Comparison and Best Model Selection
All model results:


,Model,Best_Params,CV_F1_Macro_Mean,CV_F1_Macro_Std,Test_Accuracy,Test_Precision_Fake,Test_Recall_Fake,Test_F1_Fake,Test_Precision_True,Test_Recall_True,Test_F1_True,Test_F1_Macro,Test_F1_Weighted
0,Stacking,{'clf__final_estimator__C': 10.0},0.9974,0.0002,0.9979,0.9991,0.9963,0.9977,0.9969,0.9993,0.9981,0.9979,0.9979
1,RandomForest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.9959,0.0007,0.9961,0.9983,0.9931,0.9957,0.9943,0.9986,0.9964,0.9960,0.9961
2,LinearSVC,{'clf__C': 1.0},0.9949,0.0005,0.9949,0.9977,0.9911,0.9944,0.9926,0.9981,0.9954,0.9949,0.9949
3,LogisticRegression,{'clf__C': 10.0},0.9933,0.0005,0.9931,0.9965,0.9882,0.9924,0.9903,0.9971,0.9937,0.9930,0.9931
4,MultinomialNB,{'clf__alpha': 0.1},0.9572,0.0034,0.9532,0.9545,0.9416,0.9481,0.9520,0.9627,0.9573,0.9527,0.9531
5,Baseline_Dummy,N/A,0.3532,0.0000,0.5460,0.0000,0.0000,0.0000,0.5460,1.0000,0.7063,0.3532,0.3857



Ranked models by CV Macro F1:


,Model,Best_Params,CV_F1_Macro_Mean,CV_F1_Macro_Std,Test_Accuracy,Test_Precision_Fake,Test_Recall_Fake,Test_F1_Fake,Test_Precision_True,Test_Recall_True,Test_F1_True,Test_F1_Macro,Test_F1_Weighted
0,Stacking,{'clf__final_estimator__C': 10.0},0.9974,0.0002,0.9979,0.9991,0.9963,0.9977,0.9969,0.9993,0.9981,0.9979,0.9979
1,RandomForest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.9959,0.0007,0.9961,0.9983,0.9931,0.9957,0.9943,0.9986,0.9964,0.9960,0.9961
2,LinearSVC,{'clf__C': 1.0},0.9949,0.0005,0.9949,0.9977,0.9911,0.9944,0.9926,0.9981,0.9954,0.9949,0.9949
3,LogisticRegression,{'clf__C': 10.0},0.9933,0.0005,0.9931,0.9965,0.9882,0.9924,0.9903,0.9971,0.9937,0.9930,0.9931
4,MultinomialNB,{'clf__alpha': 0.1},0.9572,0.0034,0.9532,0.9545,0.9416,0.9481,0.9520,0.9627,0.9573,0.9527,0.9531



Best model selected by CV Macro F1: Stacking


,Model,CV_F1_Macro_Mean,CV_F1_Macro_Std,Test_Accuracy,Test_F1_Fake,Test_F1_True,Test_F1_Macro,Test_F1_Weighted
0,Stacking,0.9974,0.0002,0.9979,0.9977,0.9981,0.9979,0.9979


In [17]:
print_section("MODEL COMPARISON SUMMARY TABLE")

display_cols = [
    "Model",
    "CV_F1_Macro_Mean",
    "CV_F1_Macro_Std",
    "Test_Accuracy",
    "Test_F1_Fake",
    "Test_F1_True",
    "Test_F1_Macro",
    "Test_F1_Weighted"
]

summary_table = results_df[display_cols].sort_values(by="CV_F1_Macro_Mean",ascending=False).reset_index(drop=True)

display(summary_table.round(4))

summary_table.to_csv(
    os.path.join(OUTPUT_DIR, "model_comparison_summary.csv"),
    index=False
)


print_section("BEST MODEL VS SECOND-BEST MODEL")

ranked = results_df[
    results_df["Model"] != "Baseline_Dummy"
].sort_values(
    by="CV_F1_Macro_Mean",
    ascending=False
).reset_index(drop=True)

best = ranked.iloc[0]
second = ranked.iloc[1]

cv_improvement = best["CV_F1_Macro_Mean"] - second["CV_F1_Macro_Mean"]
test_macro_improvement = best["Test_F1_Macro"] - second["Test_F1_Macro"]

best_model_comparison_text = f"""
Best model selected by CV Macro F1: {best["Model"]}
Second-best model by CV Macro F1: {second["Model"]}

Best CV Macro F1: {best["CV_F1_Macro_Mean"]:.6f}
Second-best CV Macro F1: {second["CV_F1_Macro_Mean"]:.6f}
CV Macro F1 improvement: {cv_improvement:.6f}

Best Test Macro F1: {best["Test_F1_Macro"]:.6f}
Second-best Test Macro F1: {second["Test_F1_Macro"]:.6f}
Test Macro F1 difference: {test_macro_improvement:.6f}

Best model Fake F1: {best["Test_F1_Fake"]:.6f}
Best model True F1: {best["Test_F1_True"]:.6f}

Interpretation:
The best model is selected based on CV Macro F1 from the training set, not based on the test set.
The test-set metrics are reported only for final evaluation.
Macro F1 is used because it treats Fake and True classes equally.
If the improvement over the second-best model is very small, a simpler model may be more practical.
"""

print(best_model_comparison_text)

save_text_report(
    best_model_comparison_text,
    "best_vs_second_best_model.txt"
)


MODEL COMPARISON SUMMARY TABLE


,Model,CV_F1_Macro_Mean,CV_F1_Macro_Std,Test_Accuracy,Test_F1_Fake,Test_F1_True,Test_F1_Macro,Test_F1_Weighted
0,Stacking,0.9974,0.0002,0.9979,0.9977,0.9981,0.9979,0.9979
1,RandomForest,0.9959,0.0007,0.9961,0.9957,0.9964,0.9960,0.9961
2,LinearSVC,0.9949,0.0005,0.9949,0.9944,0.9954,0.9949,0.9949
3,LogisticRegression,0.9933,0.0005,0.9931,0.9924,0.9937,0.9930,0.9931
4,MultinomialNB,0.9572,0.0034,0.9532,0.9481,0.9573,0.9527,0.9531
5,Baseline_Dummy,0.3532,0.0000,0.5460,0.0000,0.7063,0.3532,0.3857



BEST MODEL VS SECOND-BEST MODEL

Best model selected by CV Macro F1: Stacking
Second-best model by CV Macro F1: RandomForest

Best CV Macro F1: 0.997433
Second-best CV Macro F1: 0.995885
CV Macro F1 improvement: 0.001548

Best Test Macro F1: 0.997894
Second-best Test Macro F1: 0.996050
Test Macro F1 difference: 0.001844

Best model Fake F1: 0.997697
Best model True F1: 0.998090

Interpretation:
The best model is selected based on CV Macro F1 from the training set, not based on the test set.
The test-set metrics are reported only for final evaluation.
Macro F1 is used because it treats Fake and True classes equally.
If the improvement over the second-best model is very small, a simpler model may be more practical.



#### ============================================================
#### STEP 8 - Retrain best model on full training data
#### GridSearchCV(refit=True) already does this on the whole train set.
#### ============================================================

In [18]:
print_section("STEP 8 - Retrain Best Model on Full Training Data")

best_model_name = results_df_nonbaseline.iloc[0]["Model"]
best_model = best_estimators[best_model_name]

print("Best model selected by CV F1:", best_model_name)
print("\nBest model object:")
print(best_model)


STEP 8 - Retrain Best Model on Full Training Data
Best model selected by CV F1: Stacking

Best model object:
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.85, max_features=10000, min_df=3,
                                 ngram_range=(1, 2), stop_words='english',
                                 strip_accents='unicode', sublinear_tf=True)),
                ('clf',
                 StackingClassifier(cv=5,
                                    estimators=[('lr',
                                                 LogisticRegression(max_iter=3000,
                                                                    random_state=42)),
                                                ('nb',
                                                 MultinomialNB(alpha=0.5)),
                                                ('svm',
                                                 LinearSVC(random_state=42)),
                                                ('rf',
                       

#### ============================================================
#### STEP 9 - Final evaluation on test set
#### Required metrics: F1, AUC, ROC, Accuracy
#### ============================================================

In [19]:
print_section("STEP 9 - Final Evaluation on Test Set")

y_test_pred = best_model.predict(X_test)
y_test_score = get_model_scores(best_model, X_test)

# Use the same evaluation function from STEP 5 & 6
final_metrics = evaluate_model(y_test, y_test_pred)

final_accuracy = final_metrics["Test_Accuracy"]
final_f1_fake = final_metrics["Test_F1_Fake"]
final_f1_true = final_metrics["Test_F1_True"]
final_f1_macro = final_metrics["Test_F1_Macro"]
final_f1_weighted = final_metrics["Test_F1_Weighted"]

print("Best model:", best_model_name)
print("Final Test Accuracy    :", round(final_accuracy, 4))
print("Final Test Fake F1     :", round(final_f1_fake, 4))
print("Final Test True F1     :", round(final_f1_true, 4))
print("Final Test Macro F1    :", round(final_f1_macro, 4))
print("Final Test Weighted F1 :", round(final_f1_weighted, 4))

report = classification_report(
    y_test,
    y_test_pred,
    target_names=["Fake", "True"],
    digits=4,
    zero_division=0
)

print("\nClassification Report:")
print(report)

final_eval_text = f"""
Best model: {best_model_name}

Final Test Accuracy: {final_accuracy:.6f}
Final Test Fake F1: {final_f1_fake:.6f}
Final Test True F1: {final_f1_true:.6f}
Final Test Macro F1: {final_f1_macro:.6f}
Final Test Weighted F1: {final_f1_weighted:.6f}

Classification Report:
{report}
"""

save_text_report(final_eval_text, "final_classification_report.txt")

# AUC / ROC
if y_test_score is not None:
    final_auc = roc_auc_score(y_test, y_test_score)
    fpr, tpr, thresholds = roc_curve(y_test, y_test_score)

    print("Final Test AUC      :", round(final_auc, 4))

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"{best_model_name} (AUC={final_auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - Final Model")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve_final_model.png"), dpi=300)
    plt.close()
else:
    final_auc = None
    print("AUC/ROC not available for this model.")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "True"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_final_model.png"), dpi=300)
plt.close()


STEP 9 - Final Evaluation on Test Set
Best model: Stacking
Final Test Accuracy    : 0.9979
Final Test Fake F1     : 0.9977
Final Test True F1     : 0.9981
Final Test Macro F1    : 0.9979
Final Test Weighted F1 : 0.9979

Classification Report:
              precision    recall  f1-score   support

        Fake     0.9991    0.9963    0.9977      3479
        True     0.9969    0.9993    0.9981      4184

    accuracy                         0.9979      7663
   macro avg     0.9980    0.9978    0.9979      7663
weighted avg     0.9979    0.9979    0.9979      7663

Final Test AUC      : 0.9997


In [20]:
print_section("ERROR ANALYSIS - Misclassified Samples")

error_df = pd.DataFrame({
    "text": X_test.values,
    "true_label": y_test.values,
    "pred_label": y_test_pred
})

misclassified = error_df[
    error_df["true_label"] != error_df["pred_label"]
].copy()

print("Number of test samples:", len(error_df))
print("Number of misclassified samples:", len(misclassified))
print("Misclassification rate:", round(len(misclassified) / len(error_df), 6))

# Save all misclassified samples
misclassified.to_csv(
    os.path.join(OUTPUT_DIR, "misclassified_samples.csv"),
    index=False
)

# Print a few examples
print("\nExamples of misclassified samples:")
for i, row in misclassified.head(10).iterrows():
    print("=" * 100)
    print("True label:", row["true_label"], "| Predicted label:", row["pred_label"])
    print(row["text"][:800])

error_analysis_text = f"""
Error Analysis Summary

Number of test samples: {len(error_df)}
Number of misclassified samples: {len(misclassified)}
Misclassification rate: {len(misclassified) / len(error_df):.6f}

Interpretation:
Although the final F1 score is very high, we still examine misclassified samples to understand model limitations.
Potential errors may occur when fake news imitates professional news writing style, or when real news contains emotional or sensational wording.
This also shows that the model is a text classification system, not a true fact-checking system.
"""

print(error_analysis_text)
save_text_report(error_analysis_text, "error_analysis_summary.txt")


ERROR ANALYSIS - Misclassified Samples
Number of test samples: 7663
Number of misclassified samples: 16
Misclassification rate: 0.002088

Examples of misclassified samples:
True label: 0 | Predicted label: 1
women in saudi arabia were just given the right to drive many conservative men are against this because they believe in separation of the sexes and they believe women aren t smart enough to drive guess they ve never heard of danica patrick comments from saudi clerics and a sheikh are laughable saudi clerics ultimately believe female drivers undermine social values and lack the intellect to navigate the nation s roadways would you give a man with half an intellect a driving licence sheikh saad al hajari head of the saudi government s religious edict authority in the southern province of assir alleged at a lecture earlier this month saudi man threatens women drivers a saudi man was arrested for allegedly threatening to attack women drivers the interior ministry said friday following

#### ============================================================
#### Statistical component - Bootstrap confidence intervals
#### ============================================================

In [21]:
print_section("STATISTICAL COMPONENT - Bootstrap Confidence Intervals")

acc_mean, acc_low, acc_high = bootstrap_metric_ci(
    y_test,
    y_test_pred,
    accuracy_score,
    n_rounds=BOOTSTRAP_ROUNDS,
    random_state=RANDOM_STATE
)

f1_mean, f1_low, f1_high = bootstrap_metric_ci(
    y_test,
    y_test_pred,
    lambda yt, yp: f1_score(yt, yp, zero_division=0),
    n_rounds=BOOTSTRAP_ROUNDS,
    random_state=RANDOM_STATE
)

bootstrap_report = f"""
Best Model: {best_model_name}

Bootstrap {BOOTSTRAP_ROUNDS} rounds - 95% Confidence Intervals

Accuracy : mean={acc_mean:.4f}, 95% CI=({acc_low:.4f}, {acc_high:.4f})
F1 Score : mean={f1_mean:.4f}, 95% CI=({f1_low:.4f}, {f1_high:.4f})
"""

print(bootstrap_report)
save_text_report(bootstrap_report, "bootstrap_confidence_intervals.txt")



STATISTICAL COMPONENT - Bootstrap Confidence Intervals

Best Model: Stacking

Bootstrap 300 rounds - 95% Confidence Intervals

Accuracy : mean=0.9979, 95% CI=(0.9969, 0.9988)
F1 Score : mean=0.9981, 95% CI=(0.9971, 0.9989)



#### ============================================================
#### Interpretability analysis
#### For interpretation, use Logistic Regression coefficients.
#### If final model is not LogisticRegression, fit a separate LR model.
#### ============================================================

In [22]:
print_section("INTERPRETABILITY ANALYSIS")

if best_model_name == "LogisticRegression":
    interpret_model = best_model
else:
    interpret_model = Pipeline([
        ("tfidf", tfidf),
        ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ])
    interpret_model.fit(X_train, y_train)

vectorizer = interpret_model.named_steps["tfidf"]
clf = interpret_model.named_steps["clf"]

feature_names = np.array(vectorizer.get_feature_names_out())
coef = clf.coef_[0]

top_true_idx = np.argsort(coef)[-20:][::-1]
top_fake_idx = np.argsort(coef)[:20]

interpret_lines = []
interpret_lines.append("Top terms indicating TRUE news:\n")
for idx in top_true_idx:
    interpret_lines.append(f"{feature_names[idx]:<30} {coef[idx]:.6f}")

interpret_lines.append("\nTop terms indicating FAKE news:\n")
for idx in top_fake_idx:
    interpret_lines.append(f"{feature_names[idx]:<30} {coef[idx]:.6f}")

interpret_text = "\n".join(interpret_lines)
print(interpret_text)
save_text_report(interpret_text, "top_terms_interpretability.txt")



INTERPRETABILITY ANALYSIS
Top terms indicating TRUE news:

reuters                        22.333060
said                           11.802045
washington reuters             8.888638
president donald               5.037046
washington                     5.001362
wednesday                      4.515508
reuters president              4.389998
tuesday                        4.334400
thursday                       4.086894
friday                         3.777070
nov                            3.694763
republican                     3.440195
monday                         3.435822
minister                       3.269880
told reuters                   3.212337
said statement                 3.205198
president barack               3.011659
york reuters                   2.838996
spokesman                      2.790199
year                           2.780441

Top terms indicating FAKE news:

read                           -6.071672
image                          -5.825136
featured image        

In [23]:
print_section("DATASET BIAS CHECK - Subject and Label Relationship")

bias_report_lines = []

if "subject" in df.columns:
    subject_counts = df["subject"].value_counts()
    subject_label_table = pd.crosstab(
        df["subject"],
        df["label"],
        normalize="index"
    )

    print("Subject counts:")
    print(subject_counts)

    print("\nSubject-label distribution:")
    print(subject_label_table)

    subject_counts.to_csv(
        os.path.join(OUTPUT_DIR, "subject_counts.csv")
    )

    subject_label_table.to_csv(
        os.path.join(OUTPUT_DIR, "subject_label_distribution.csv")
    )

    bias_report_lines.append("Dataset Bias Check: Subject and Label Relationship")
    bias_report_lines.append("=" * 70)
    bias_report_lines.append("")
    bias_report_lines.append("This analysis checks whether some subjects are strongly associated with one label.")
    bias_report_lines.append("If subject categories are highly correlated with fake or real labels,")
    bias_report_lines.append("the model may learn topic/source-specific shortcuts instead of general fake-news patterns.")
    bias_report_lines.append("")
    bias_report_lines.append("For this reason, the main experiment uses title + text only and excludes subject from text features.")
    bias_report_lines.append("")
    bias_report_lines.append("Subject counts:")
    bias_report_lines.append(subject_counts.to_string())
    bias_report_lines.append("")
    bias_report_lines.append("Subject-label distribution:")
    bias_report_lines.append(subject_label_table.to_string())

else:
    print("No subject column found. Skipping subject bias check.")
    bias_report_lines.append("No subject column found. Subject bias check skipped.")

bias_report = "\n".join(bias_report_lines)
save_text_report(bias_report, "subject_bias_check.txt")


DATASET BIAS CHECK - Subject and Label Relationship
Subject counts:
subject
politicsNews       11212
worldnews           9709
News                9050
politics            6366
US_News              783
left-news            679
Government News      514
Name: count, dtype: int64

Subject-label distribution:
label              0    1
subject                  
Government News  1.0  0.0
News             1.0  0.0
US_News          1.0  0.0
left-news        1.0  0.0
politics         1.0  0.0
politicsNews     0.0  1.0
worldnews        0.0  1.0


#### ============================================================
#### Save final summary
#### ============================================================

In [24]:
print_section("SAVE FINAL SUMMARY")

summary_lines = []
summary_lines.append("Fake News Detection - Final Summary")
summary_lines.append("=" * 70)
summary_lines.append(f"Feature mode: {FEATURE_MODE}")
summary_lines.append(f"Training samples: {len(X_train)}")
summary_lines.append(f"Testing samples : {len(X_test)}")
summary_lines.append("")
summary_lines.append("Model selection rule:")
summary_lines.append("The best model is selected by CV Macro F1 on the training set, not by test-set performance.")
summary_lines.append("The test set is used only for final evaluation.")
summary_lines.append("")
summary_lines.append("All model results:")
summary_lines.append(results_df.to_string(index=False))
summary_lines.append("")

best_row = results_df[
    results_df["Model"] == best_model_name
].iloc[0]

summary_lines.append(f"Best model selected by CV Macro F1: {best_model_name}")
summary_lines.append(f"Best CV Macro F1 Mean: {best_row['CV_F1_Macro_Mean']:.6f}")
summary_lines.append(f"Best CV Macro F1 Std : {best_row['CV_F1_Macro_Std']:.6f}")
summary_lines.append("")
summary_lines.append("Final test metrics:")
summary_lines.append(f"Accuracy    : {final_accuracy:.4f}")
summary_lines.append(f"Fake F1     : {final_f1_fake:.4f}")
summary_lines.append(f"True F1     : {final_f1_true:.4f}")
summary_lines.append(f"Macro F1    : {final_f1_macro:.4f}")
summary_lines.append(f"Weighted F1 : {final_f1_weighted:.4f}")

if final_auc is not None:
    summary_lines.append(f"AUC         : {final_auc:.4f}")

summary_lines.append("")
summary_lines.append("Classification Report:")
summary_lines.append(report)
summary_lines.append("")
summary_lines.append("Bootstrap Confidence Intervals:")
summary_lines.append(bootstrap_report)

summary_lines.append("")
summary_lines.append("Additional Analysis:")
summary_lines.append("- Model comparison summary table saved as model_comparison_summary.csv")
summary_lines.append("- Best vs second-best model analysis saved as best_vs_second_best_model.txt")
summary_lines.append("- Misclassified samples saved as misclassified_samples.csv")
summary_lines.append("- Error analysis summary saved as error_analysis_summary.txt")
summary_lines.append("- Subject bias check saved as subject_bias_check.txt")
summary_lines.append("")

summary_lines.append("Limitations:")
summary_lines.append("1. This model is a text classification system, not a real fact-checking system.")
summary_lines.append("2. It learns statistical patterns from labeled data, such as wording, style, and structure.")
summary_lines.append("3. It cannot directly verify whether a factual claim is objectively true.")
summary_lines.append("4. Very high performance may be partly related to dataset-specific patterns or source/style bias.")
summary_lines.append("5. Since the dataset only contains binary labels, the current model cannot fully handle partially true or mixed news.")
summary_lines.append("")

summary_lines.append("Future Work:")
summary_lines.append("1. Test the model on external datasets and newer news articles.")
summary_lines.append("2. Use source-based or time-based splits to evaluate generalization.")
summary_lines.append("3. Add more detailed error analysis for ambiguous or partially misleading news.")
summary_lines.append("4. Improve traditional text representations using TF-IDF tuning, feature selection, and SVD/LSA.")
summary_lines.append("5. Explore metadata such as source credibility, author information, or social engagement patterns.")

final_summary = "\n".join(summary_lines)
save_text_report(final_summary, "final_project_summary.txt")

print("All results saved to folder:", OUTPUT_DIR)
print("Done.")


SAVE FINAL SUMMARY
All results saved to folder: final_results_fake_news_v4
Done.
